# X-REACH — PPO Training (5-fold, per-fold evaluation, backend-ready export)

Trains PPO on the 5 folds produced by `X-REACH_Full_Preprocessing_Pipeline.ipynb`
(`rl_environment.pkl`), evaluates every fold's own metrics separately, then
lets you download the best fold's trained model in the exact format the
X-REACH backend expects.

Includes the two fixes discussed earlier for the currently deployed PPO
artifact always showing exactly 0% or 100% confidence:
- `ENTROPY_COEFFICIENT` raised from 0.01 to 0.05 (still constant throughout
  training, just stronger), plus dropout added to the actor/critic heads,
  to resist the policy collapsing to near-deterministic outputs.
- The export cell records `runtime_compatibility.output_activation:
  "softmax"` in `metadata.json` -- PPO's actor head already applies its own
  softmax, so the backend must use its output as-is rather than re-applying
  softmax (that double-softmax was the root cause of the 0%/100% bug).

**How to use:** Runtime -> Run all. When prompted, upload `rl_environment.pkl`
(from the preprocessing notebook). At the end it triggers a browser download
of `weights.json` and `metadata.json` for the best fold -- no ONNX, no
`onnxruntime-node`, no PyTorch needed anywhere near the backend.


In [ ]:
import os
import random
import pickle
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Load the 5-fold RL environment

Each fold is `{fold, states_train, labels_train, states_test, labels_test,
state_size, action_size}`. `states_train` already includes the augmented
rows (if you ran the preprocessing notebook); `states_test` is always real,
unaugmented data.


In [ ]:
ENV_PATH = "rl_environment.pkl"

if not os.path.exists(ENV_PATH):
    try:
        from google.colab import files
        print("Please upload rl_environment.pkl")
        uploaded = files.upload()
        ENV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{ENV_PATH} not found. Place it next to this notebook.")

with open(ENV_PATH, "rb") as f:
    folds = pickle.load(f)

print(f"Loaded {len(folds)} folds")
for fold in folds:
    fold_number = fold["fold"]
    train_shape = fold["states_train"].shape
    test_shape = fold["states_test"].shape
    print(f"  Fold {fold_number}: states_train {train_shape}, states_test {test_shape}, state_size {fold['state_size']}")

STATE_SIZE = folds[0]["state_size"]

# Canonical feature order (see backend/utils/featureContract.js) -- for metadata.json only.
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs",
]


## Hyperparameters & network

Same architecture as the currently deployed model
(`shared(15->128->128) + actor(128->128->2, softmax) + critic(128->128->1)`),
with **dropout** added to the shared/actor layers and `ENTROPY_COEFFICIENT`
raised from `0.01` to `0.05`. Both are aimed at the same problem: the
original 0.01 entropy bonus wasn't strong enough, relative to how many
gradient updates 500 episodes x 10 PPO epochs produces, to stop the actor's
output logits from growing large enough to saturate softmax to
near-exact 0/1.


In [ ]:
PPO_NUM_EPISODES = 500
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPSILON = 0.2
ENTROPY_COEFFICIENT = 0.05   # CHANGED from 0.01 -- stronger, sustained entropy pressure
VALUE_LOSS_COEFFICIENT = 0.5
PPO_EPOCHS = 10
BATCH_SIZE = 64
ADAM_EPSILON = 1e-5
MAX_GRAD_NORM = 0.5
HIDDEN_DIM = 128
ACTION_SIZE = 2
DROPOUT = 0.15   # NEW


class HeartDiseaseEnvironment:
    """Same reward structure as the original notebook (correct
    classification = +1, incorrect = -1), but shuffles training row order
    every episode (NEW -- the original always walked states_train starting
    at index 0 in the same fixed order every single episode)."""
    def __init__(self, fold_data, seed):
        self.states_train = fold_data["states_train"]
        self.labels_train = fold_data["labels_train"]
        self.rng = np.random.RandomState(seed)
        self.order = None
        self.pos = 0

    def reset(self):
        self.order = self.rng.permutation(len(self.states_train))
        self.pos = 0
        return self.states_train[self.order[self.pos]]

    def step(self, action):
        true_label = self.labels_train[self.order[self.pos]]
        reward = 1 if action == true_label else -1
        self.pos += 1
        done = self.pos >= len(self.order)
        next_state = self.states_train[self.order[self.pos]] if not done else self.states_train[self.order[-1]]
        return next_state, reward, done, {}


class ActorCritic(nn.Module):
    def __init__(self, state_size):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_size, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
        )
        self.actor = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, ACTION_SIZE), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        self.to(DEVICE)

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state)
        state = state.to(DEVICE)
        features = self.shared(state)
        return self.actor(features), self.critic(features)

    def act(self, state):
        action_probs, value = self.forward(state)
        dist = Categorical(action_probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action).detach(), value.detach()

    def evaluate(self, states, actions):
        action_probs, values = self.forward(states)
        dist = Categorical(action_probs)
        return dist.log_prob(actions), values.squeeze(), dist.entropy()


class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states, self.actions, self.rewards = [], [], []
        self.dones, self.log_probs, self.values = [], [], []

    def store(self, state, action, reward, done, log_prob, value):
        self.states.append(state); self.actions.append(action)
        self.rewards.append(reward); self.dones.append(done)
        self.log_probs.append(log_prob); self.values.append(value)

    def __len__(self):
        return len(self.states)

    def compute_returns_and_advantages(self):
        values = [v.item() for v in self.values]
        returns, advantages = [], []
        gae, next_value = 0.0, 0.0
        for t in reversed(range(len(self.rewards))):
            mask = 1.0 - float(self.dones[t])
            delta = self.rewards[t] + GAMMA * next_value * mask - values[t]
            gae = delta + GAMMA * GAE_LAMBDA * mask * gae
            advantages.insert(0, gae)
            next_value = values[t]
        returns = [a + v for a, v in zip(advantages, values)]
        self.returns = torch.FloatTensor(returns).to(DEVICE)
        self.advantages = torch.FloatTensor(advantages).to(DEVICE)
        self.states_t = torch.FloatTensor(np.array(self.states)).to(DEVICE)
        self.actions_t = torch.LongTensor(self.actions).to(DEVICE)
        self.log_probs_t = torch.FloatTensor(self.log_probs).to(DEVICE)

    def normalize_advantages(self):
        self.advantages = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)

    def generate_batches(self):
        n = len(self.states)
        indices = np.random.permutation(n)
        for start in range(0, n, BATCH_SIZE):
            batch_idx = indices[start:start + BATCH_SIZE]
            yield (self.states_t[batch_idx], self.actions_t[batch_idx],
                   self.log_probs_t[batch_idx], self.returns[batch_idx], self.advantages[batch_idx])


class PPOAgent:
    def __init__(self, state_size):
        self.policy = ActorCritic(state_size).to(DEVICE)
        self.old_policy = ActorCritic(state_size).to(DEVICE)
        self.old_policy.load_state_dict(self.policy.state_dict())
        self.buffer = RolloutBuffer()
        self.optimizer = optim.Adam(self.policy.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON)
        self.mse_loss = nn.MSELoss()

    def select_action(self, state):
        with torch.no_grad():
            return self.old_policy.act(state)

    def store_transition(self, **kwargs):
        self.buffer.store(**kwargs)

    def update_policy(self):
        if len(self.buffer) == 0:
            return
        self.buffer.compute_returns_and_advantages()
        self.buffer.normalize_advantages()

        for _ in range(PPO_EPOCHS):
            for states, actions, old_log_probs, returns, advantages in self.buffer.generate_batches():
                new_log_probs, state_values, entropy = self.policy.evaluate(states, actions)
                ratio = torch.exp(new_log_probs - old_log_probs)
                clipped_ratio = torch.clamp(ratio, 1 - CLIP_EPSILON, 1 + CLIP_EPSILON)
                policy_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()
                value_loss = self.mse_loss(state_values, returns)
                entropy_bonus = entropy.mean()
                # ENTROPY_COEFFICIENT is the key fix -- see cell markdown above.
                total_loss = policy_loss + VALUE_LOSS_COEFFICIENT * value_loss - ENTROPY_COEFFICIENT * entropy_bonus

                self.optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), MAX_GRAD_NORM)
                self.optimizer.step()

        self.old_policy.load_state_dict(self.policy.state_dict())
        self.buffer.clear()


def train_ppo_episode(env, agent):
    state, done, episode_reward = env.reset(), False, 0
    while not done:
        action, log_prob, value = agent.select_action(state)
        next_state, reward, done, _ = env.step(action)
        agent.store_transition(state=state, action=action, reward=reward, done=done,
                                log_prob=log_prob, value=value)
        state = next_state
        episode_reward += reward
    agent.update_policy()
    return episode_reward


## Train all 5 folds

Each fold gets its own freshly-initialized agent, trained only on that
fold's `states_train` (which already includes the augmented rows), and
evaluated only on that fold's own `states_test` (always real data).


In [ ]:
def train_ppo_fold(fold_data, seed):
    env = HeartDiseaseEnvironment(fold_data, seed=seed)
    agent = PPOAgent(STATE_SIZE)
    agent.policy.train()
    for episode in range(PPO_NUM_EPISODES):
        reward = train_ppo_episode(env, agent)
        if (episode + 1) % 100 == 0:
            print(f"    episode {episode+1}/{PPO_NUM_EPISODES} | reward {reward}")
    return agent


print("=" * 70)
print(f"TRAINING PPO -- {len(folds)} folds")
print("=" * 70)

ppo_agents = []
for i, fold_data in enumerate(folds):
    print(f"\nFold {fold_data['fold']}")
    agent = train_ppo_fold(fold_data, seed=SEED + i)
    ppo_agents.append(agent)

print("\nPPO training complete for all folds.")


## Evaluate each fold separately

Prints and stores per-fold Accuracy / Precision / Recall / F1 / Specificity
/ ROC-AUC / confusion matrix, exactly like the original notebook's
evaluation cell, plus saves `ppo_evaluation_results.csv`.


In [ ]:
all_metrics = []

print("=" * 70)
print("EVALUATING EACH FOLD")
print("=" * 70)

for fold_data, agent in zip(folds, ppo_agents):
    fold_number = fold_data["fold"]
    agent.policy.eval()

    X_test = np.array(fold_data["states_test"], dtype=np.float32)
    y_test = np.array(fold_data["labels_test"], dtype=np.int64)

    predictions, probabilities = [], []
    with torch.no_grad():
        for state in X_test:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            action_probs, _ = agent.policy(state_tensor)
            pred = torch.argmax(action_probs, dim=1).item()
            predictions.append(pred)
            probabilities.append(action_probs[0, 1].item())

    agent.policy.train()

    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions, zero_division=0)
    rec = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    cm = confusion_matrix(y_test, predictions)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0

    try:
        auc = roc_auc_score(y_test, probabilities)
    except ValueError:
        auc = float("nan")

    metrics = {"Fold": fold_number, "Accuracy": acc, "Precision": prec, "Recall": rec,
               "F1": f1, "Specificity": specificity, "ROC_AUC": auc}
    all_metrics.append(metrics)

    print(f"\nFold {fold_number}")
    print(metrics)
    print("Confusion Matrix:")
    print(cm)

results_df = pd.DataFrame(all_metrics)
print("\n" + "=" * 70)
print("AVERAGE PERFORMANCE ACROSS ALL FOLDS")
print("=" * 70)
print(results_df.mean(numeric_only=True))

results_df.to_csv("ppo_evaluation_results.csv", index=False)
print("\nSaved ppo_evaluation_results.csv")
results_df


## Sanity check -- confidence values are no longer pinned to 0%/100%

Quick check before exporting: this prints the actual predicted probability
for a handful of test-set patients from the best fold. If the entropy fix
worked, these should be spread out, not clustered at exactly 0.0/1.0 like
the currently deployed artifact.


In [ ]:
best_idx_preview = int(results_df["Accuracy"].idxmax())
preview_agent = ppo_agents[best_idx_preview]
preview_agent.policy.eval()

X_test = np.array(folds[best_idx_preview]["states_test"], dtype=np.float32)
with torch.no_grad():
    sample_probs = []
    for state in X_test[:15]:
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        action_probs, _ = preview_agent.policy(state_tensor)
        sample_probs.append(round(action_probs[0, 1].item(), 4))

preview_agent.policy.train()
print("Sample predicted probabilities (first 15 test patients, best fold):")
print(sample_probs)
print(f"\nDistinct values: {len(set(sample_probs))} / {len(sample_probs)} "
      f"(higher is better -- the old artifact would show ~2 distinct values here)")


## Export the best fold for the backend (no ONNX)

Instead of exporting to ONNX, this dumps the trained actor network's
weights as plain JSON -- a list of layers (`linear` / `relu` / `softmax`),
each `linear` layer giving its weight matrix and bias vector as plain
nested arrays -- plus `metadata.json`. Only `shared` + `actor` are
exported (the critic head is only needed during training, not for
predictions). Dropout layers are skipped (a no-op at inference time). This
is the same pattern already used for Q-Learning in this backend
(`models/rl/qlearning-2026-08-13/model.json` +
`backend/services/adapters/qtableAdapter.js`) -- the backend does the
matrix multiplications itself in plain JavaScript, so it needs **no
PyTorch, no ONNX, and no `onnxruntime-node` dependency at all**.

The softmax is already included as the *last* layer in the exported list
(matching the actor head's own architecture) -- the backend must use the
final layer's output as-is, not apply softmax again (that double-softmax
was the root cause of the original 0%/100% saturation bug).

The cell below also verifies the exported JSON reproduces the exact same
output as the live PyTorch model on random inputs, before you download it.

**Give me the downloaded `best_model_ppo/` folder's two files**
(`weights.json`, `metadata.json`) and I'll place them at
`models/rl/ppo-fold<N>-<date>/` in the backend and write the matching JS
adapter -- no other backend changes needed since the scaler parameters are
unchanged.


In [ ]:
BEST_MODEL_DIR = "best_model_ppo"
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

best_idx = int(results_df["Accuracy"].idxmax())
best_fold_number = int(results_df.iloc[best_idx]["Fold"])
best_row = results_df.iloc[best_idx]
best_agent = ppo_agents[best_idx]

print("=" * 70)
print("BEST FOLD SELECTION")
print("=" * 70)
print(results_df[["Fold", "Accuracy", "ROC_AUC"]].to_string(index=False))
print(f"\nBest fold: {best_fold_number} (accuracy {best_row['Accuracy']:.4f})")

model_path = os.path.join(BEST_MODEL_DIR, "model.pth")
torch.save({
    "actor_state_dict": best_agent.policy.state_dict(),
    "critic_state_dict": best_agent.policy.critic.state_dict(),
}, model_path)

metadata = {
    "model_version": f"ppo-fold{best_fold_number}-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}",
    "algorithm": "PPO",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "preprocessing": {
        "note": "Trained on the 15-dimension representation produced by backend/services/preprocessingService.js. Architecture: shared(15->128->128) + actor(128->128->2, softmax) + critic(128->128->1), with dropout added.",
        "trained_state_dimension": STATE_SIZE,
    },
    "training_metadata": {
        "fold": best_fold_number,
        "num_folds": len(folds),
        "episodes": PPO_NUM_EPISODES,
        "entropy_coefficient": ENTROPY_COEFFICIENT,
        "dropout": DROPOUT,
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    },
    "evaluation_metadata": {
        "accuracy": float(best_row["Accuracy"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Recall"]),
        "f1": float(best_row["F1"]),
        "roc_auc": None if pd.isna(best_row["ROC_AUC"]) else float(best_row["ROC_AUC"]),
    },
    "runtime_compatibility": {
        "framework": "javascript",
        "export_format": "layers-json",   # distinct from Q-Learning's "json" (q-table) -- this is a layer-list neural net
        "weights_file": "weights.json",
        "input_dim": STATE_SIZE,
        "num_classes": 2,
        "output_activation": "softmax",   # already applied as the LAST layer in weights.json -- do not re-apply
    },
}


def export_layers_to_json(*sequential_modules):
    """Walks one or more nn.Sequential modules IN ORDER and emits a plain
    JSON-serializable layer list. Dropout is skipped (a no-op at eval()
    time -- contributes nothing to inference, no JS equivalent needed)."""
    layers = []
    for sequential_module in sequential_modules:
        for module in sequential_module:
            if isinstance(module, nn.Linear):
                layers.append({
                    "type": "linear",
                    # weight is [out_features][in_features] (PyTorch's own layout) --
                    # the JS forward pass does out[i] = bias[i] + sum_j weight[i][j] * x[j]
                    "weight": module.weight.detach().cpu().numpy().tolist(),
                    "bias": module.bias.detach().cpu().numpy().tolist(),
                })
            elif isinstance(module, nn.ReLU):
                layers.append({"type": "relu"})
            elif isinstance(module, nn.Softmax):
                layers.append({"type": "softmax"})
            elif isinstance(module, nn.Dropout):
                continue   # inference-time no-op, deliberately not exported
            else:
                raise ValueError(f"Unsupported layer type for JSON export: {type(module)}")
    return layers


best_agent.policy.eval()
# shared + actor only -- the critic head is training-only, not needed for predictions.
ppo_layers = export_layers_to_json(best_agent.policy.shared, best_agent.policy.actor)

weights_path = os.path.join(BEST_MODEL_DIR, "weights.json")
with open(weights_path, "w") as f:
    json.dump({"input_dim": STATE_SIZE, "layers": ppo_layers}, f)

metadata_path = os.path.join(BEST_MODEL_DIR, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved: {weights_path}")
print(f"Saved: {metadata_path}")


# --- Verify the exported JSON reproduces the live model exactly ---------
def numpy_forward(layers, x):
    for layer in layers:
        if layer["type"] == "linear":
            W = np.array(layer["weight"])   # [out, in]
            b = np.array(layer["bias"])     # [out]
            x = W @ x + b
        elif layer["type"] == "relu":
            x = np.maximum(0, x)
        elif layer["type"] == "softmax":
            x = np.exp(x - x.max())
            x = x / x.sum()
    return x


with torch.no_grad():
    for _ in range(5):
        sample = np.random.randn(STATE_SIZE).astype(np.float32)
        torch_out, _ = best_agent.policy(torch.FloatTensor(sample).unsqueeze(0))
        torch_out = torch_out.cpu().numpy()[0]
        json_out = numpy_forward(ppo_layers, sample.astype(np.float64))
        max_diff = np.abs(torch_out - json_out).max()
        assert max_diff < 1e-4, f"Exported JSON weights diverge from the live model! max diff = {max_diff}"

print("Verified: exported JSON weights reproduce the live PyTorch model's output exactly.")

try:
    from google.colab import files
    files.download(weights_path)
    files.download(metadata_path)
    print("\nDownload triggered for both files.")
except ImportError:
    print(f"\nNot in Colab -- files are available locally under {BEST_MODEL_DIR}/")
